# Composer v43 · Colab Pro / Pro+ 续训

选择 **GPU + 高 RAM**，优先 A100/L4 等支持 BF16 的 GPU。保留原训练/验证集、学习合同、优化器和进度，16×4、固定学习率 0.0005、总目标 50 轮。

将 `composer_colab.tar.gz` 和 `.sha256`（或全部分片、分片清单和 `.sha256`）上传到下面的 Drive 目录。本笔记本不含数据和权重，也不公开它们。运行时重建后从 Drive 的最新完整备份恢复；断线时尚未备份的更新需重跑。

先执行 100 次更新测速；确认日志中数据指纹验证通过，再执行正式训练。Colab GPU/高 RAM 的供应不保证。跨平台不保证逐位相同，数据合同不匹配时训练器会拒绝续训，不应绕过校验。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_HOME = Path('/content/drive/MyDrive/Composer_Colab_20261102')  # 修改为上传所在目录
ARCHIVE = DRIVE_HOME / 'composer_colab.tar.gz'
RUN_BACKUP = DRIVE_HOME / 'runs/v43_colab'
BUNDLE = Path('/content/composer_bundle')
LOCAL_RUN = Path('/content/composer_run_v43')


In [ ]:
import torch, shutil, os
assert torch.cuda.is_available(), '请先在运行时设置中选择 GPU'
print('GPU:', torch.cuda.get_device_name())
assert torch.cuda.get_device_capability()[0] >= 8, '当前 GPU 不支持本项目的原生 BF16；请选择 A100/L4 等 GPU'
print('RAM GiB:', round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30, 1))
print('Disk free GiB:', round(shutil.disk_usage('/content').free / 2**30, 1))
assert os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') >= 30 * 2**30, '请改用高 RAM 运行时，建议至少 32 GiB'


In [ ]:
import hashlib, tarfile
assert ARCHIVE.is_file(), f'请先上传 {ARCHIVE.name} 到 {DRIVE_HOME}'
checksum_file = ARCHIVE.with_suffix(ARCHIVE.suffix + '.sha256')
expected = checksum_file.read_text().split()[0]
assert shutil.disk_usage('/content').free > ARCHIVE.stat().st_size + 17 * 2**30, '本地磁盘不足'
local_archive = Path('/content/composer_colab.tar.gz')
shutil.copyfile(ARCHIVE, local_archive)
h = hashlib.sha256()
with local_archive.open('rb') as stream:
    for block in iter(lambda: stream.read(8*1024*1024), b''):
        h.update(block)
assert h.hexdigest() == expected, '迁移包校验失败，请重新上传'
assert not BUNDLE.exists(), '此运行时已解压；重新连接后可跳过本格，避免混合两个迁移包'
BUNDLE.mkdir()
with tarfile.open(local_archive) as archive:
    archive.extractall(BUNDLE, filter='data')
local_archive.unlink()
print('数据和缓存已放到 Colab 本地磁盘')


In [ ]:
import hashlib, tarfile, json
parts_manifest = ARCHIVE.with_name('composer_colab.parts.json')
if ARCHIVE.is_file():
    sources = [ARCHIVE]
else:
    assert parts_manifest.is_file(), f'请先上传迁移包或全部分片到 {DRIVE_HOME}'
    parts = json.loads(parts_manifest.read_text())['parts']
    sources = [DRIVE_HOME / entry['name'] for entry in parts]
    assert all(path.is_file() for path in sources), '分片尚未上传完整'
    assert all(path.stat().st_size == entry['size'] for path, entry in zip(sources, parts)), '分片大小不符'
checksum_file = ARCHIVE.with_suffix(ARCHIVE.suffix + '.sha256')
expected = checksum_file.read_text().split()[0]
assert shutil.disk_usage('/content').free > sum(p.stat().st_size for p in sources) + 17 * 2**30, '本地磁盘不足'
local_archive = Path('/content/composer_colab.tar.gz')
with local_archive.open('wb') as target:
    for source in sources:
        print('读取:', source.name)
        with source.open('rb') as stream:
            shutil.copyfileobj(stream, target, length=8*1024*1024)
h = hashlib.sha256()
with local_archive.open('rb') as stream:
    for block in iter(lambda: stream.read(8*1024*1024), b''):
        h.update(block)
assert h.hexdigest() == expected, '迁移包校验失败，请重新上传'
assert not BUNDLE.exists(), '此运行时已解压；重新连接后可跳过本格，避免混合两个迁移包'
BUNDLE.mkdir()
with tarfile.open(local_archive) as archive:
    archive.extractall(BUNDLE, filter='data')
local_archive.unlink()
print('数据和缓存已放到 Colab 本地磁盘')


In [ ]:
def train(updates):
    command = [sys.executable, '-u', str(PROJECT / 'tools/colab_resume.py'),
               '--bundle', str(BUNDLE), '--output', str(LOCAL_RUN),
               '--backup-dir', str(RUN_BACKUP), '--updates', str(updates), '--data-workers', '0']
    # 默认共享线程，先确认实际 RAM 余量；内存充足时可对照测试 data-workers=1。
    log = RUN_BACKUP / 'training.log'
    with log.open('a', encoding='utf-8') as stream:
        process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='')
                stream.write(line)
                stream.flush()
            if process.wait():
                raise RuntimeError(f'训练退出码 {process.returncode}，请检查上方日志')
        finally:
            if process.poll() is None:
                process.terminate()
                process.wait()
            for name in ('validation_latest.json',):
                if (LOCAL_RUN / name).is_file():
                    shutil.copyfile(LOCAL_RUN / name, RUN_BACKUP / name)
            for panel in LOCAL_RUN.glob('generation_eval_epoch_*'):
                if panel.is_dir():
                    shutil.copytree(panel, RUN_BACKUP / panel.name, dirs_exist_ok=True)

train(100)  # 从完整点继续 100 次更新，保存到独立云端目录


比较同一执行中两个普通训练日志点的 `effective_target_budget` 差 / `elapsed_sec` 差，同时看 `data_sec` 和 `compute_sec`。首次数据准备、恢复、保存、验证和生成抽检单独计算。模型质量须另看固定歌曲、双种子和试玩；更快训练不是质量改善的证据。

确认测速和 Drive 备份成功后，执行下一格。保留本机模型；云端输出位于独立目录，不会自动合并回本机。每 100 次更新同步备份完整检查点及推理权重；如写入 Drive 失败会报错停止。


In [ ]:
train(0)  # 续训到总目标 50 轮，不是额外训练 50 轮
